In [ ]:
import requests
import json

In [ ]:
!powershell "az account set --subscription a8330230-b8a0-4839-8650-17faf7ddcc42"

In [ ]:
!powershell "git checkout dataphos-workshops"
!powershell "git pull"

In [ ]:
participant_identification = ""

In [ ]:
if not participant_identification.islower() or not participant_identification.isalpha():
    raise ValueError("The participant identification prefix must be lowercase, without any whitespaces or special characters.")

with open("Pulumi.workshop-participant-config.yaml", 'r') as file:
    content = file.read()

content = content.replace("<participant_identification>", participant_identification)

with open("Pulumi.workshop-participant-config.yaml", 'w') as file:
    file.write(content)

In [ ]:
!powershell "pulumi stack init workshop-participant-config"

In [ ]:
!powershell "pulumi preview"

In [ ]:
!powershell "pulumi up --yes"

In [ ]:
schema_registry_service_ip = ""

In [ ]:
url = "http://" + schema_registry_service_ip + ":8080/schemas"
headers = {"content-type": "application/json", "Accept-Charset": "UTF-8"}

r = requests.get(url, headers=headers)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
schema = {
    "description": "Schema registered manually for testing Dataphos.",
    "schema_type": "json",
    "specification": '{\r\n   "$schema":"https:\/\/json-schema.org\/draft-07\/schema",\r\n   "additionalProperties":false,\r\n   "type":"object",\r\n   "properties":{\r\n      "creation_timestamp":{\r\n         "type":"string"\r\n      },\r\n      "customer_info":{\r\n         "type":"object",\r\n         "properties":{\r\n            "age":{\r\n               "type":"integer"\r\n            },\r\n            "customer_id":{\r\n               "type":"integer"\r\n            },\r\n            "name":{\r\n               "type":"string"\r\n            },\r\n            "surname":{\r\n               "type":"string"\r\n            }\r\n         },\r\n         "required":[\r\n            "age",\r\n            "customer_id",\r\n            "name",\r\n            "surname"\r\n         ]\r\n      },\r\n      "invoice_id":{\r\n         "type":"integer"\r\n      },\r\n      "items":{\r\n         "type":"array",\r\n         "items":{\r\n            "type":"object",\r\n            "properties":{\r\n               "item_name":{\r\n                  "type":"string"\r\n               },\r\n               "price":{\r\n                  "type":"string"\r\n               }\r\n            },\r\n            "required":[\r\n               "item_name",\r\n               "price"\r\n            ]\r\n         }\r\n      }\r\n   },\r\n   "required":[\r\n      "creation_timestamp",\r\n      "customer_info",\r\n      "invoice_id",\r\n      "items"\r\n   ]\r\n}',
    "name": "Manually registered schema",
    "publisher_id": "",
    "compatibility_mode": "backward",
    "validity_mode": "full",
}

r = requests.post(url, headers=headers, json=schema)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
resubmitter_service_ip = ""

In [ ]:
resubmitter_settings = {
    "broker_id": participant_identification + "-valid-topic",
    "lb": "2024-08-06T09:45:00Z"
}
url = "http://" + resubmitter_service_ip + ":8081/range/indexer_collection?topic=" + participant_identification + "-resubmitter-topic"

r = requests.post(url, headers=headers, json=resubmitter_settings)
print("Satus code of the response: ", r.status_code)
print(json.dumps(r.json(), indent=4))

In [ ]:
!powershell "pulumi destroy --yes"